# 🧠 Traditional RAG — A Crash Course for Beginners

> **Learn how to build a Retrieval-Augmented Generation (RAG) pipeline from scratch using Python, LangChain, ChromaDB and Groq.**

This notebook walks you through every step of a traditional RAG pipeline interactively.

---
## 📦 Setup & Configuration

Load environment variables and set up paths.

In [1]:
import os
from dotenv import load_dotenv

load_dotenv()

# Directory paths
PERSIST_DIRECTORY = "../data/vector_store"
PDF_DIR = "../data/pdfs"
TEXT_DIR = "../data/text_files"

# Ensure directories exist
os.makedirs(PERSIST_DIRECTORY, exist_ok=True)
os.makedirs(PDF_DIR, exist_ok=True)
os.makedirs(TEXT_DIR, exist_ok=True)

print("✅ Configuration loaded.")

✅ Configuration loaded.


---
## 📄 Understanding LangChain Documents

Before we load real files, let's understand the **Document** object — the fundamental data structure in LangChain.

Every loaded file becomes a `Document` with:
- `page_content` — the actual text
- `metadata` — information about the source (filename, page number, etc.)

In [2]:
### Document Structure
from langchain_core.documents import Document

doc = Document(
    page_content="This is a document content",
    metadata={
        "source": "example.txt",
        "page": 1,
        "author": "DevZees",
        "created_date": "2026-07-30"
    }
)

In [3]:
doc

Document(metadata={'source': 'example.txt', 'page': 1, 'author': 'DevZees', 'created_date': '2026-07-30'}, page_content='This is a document content')

---
## 📝 Create Sample Text Files

Let's create some sample `.txt` files to work with in our pipeline.

In [4]:
## Create sample txt files
import os
os.makedirs("../data/text_files", exist_ok=True)

simple_texts = {
    "../data/text_files/python_intro.txt": """Python is a high-level, interpreted, general-purpose programming language created by Guido van Rossum and first released in 1991. It is known for its simple, readable syntax, making it one of the easiest languages to learn and one of the most widely used in software development.

Why Python is popular
Easy to read and write
Large standard library
Huge ecosystem of third-party packages
Cross-platform (Windows, Linux, macOS)
Excellent community support
Suitable for beginners and professionals alike
    """,
        "../data/text_files/java_intro.txt": """Java is a high-level, object-oriented, class-based programming language developed by James Gosling at Sun Microsystems and first released in 1995. Today, Java is owned and maintained by Oracle Corporation.

Java is one of the most popular languages for building enterprise applications, web services, mobile apps, cloud-native systems, and large-scale distributed applications.

Key Features
Platform-independent ("Write Once, Run Anywhere")
Object-Oriented Programming (OOP)
Strongly and statically typed
Automatic memory management (Garbage Collection)
Robust exception handling
Multithreading support
Rich standard library
    """
}



for filepath, content in simple_texts.items():
    with open(filepath, 'w', encoding="utf-8") as f:
        f.write(content)

print("✅ Sample text file created.")

✅ Sample text file created.


---
## Step 1 — 📥 Data Ingestion (Loading Your Documents)

> **Goal:** Read raw files (PDFs & text) and convert them into a format Python can work with.

- `PyPDFDirectoryLoader` reads every PDF in the `data/pdfs/` folder and extracts the text page by page.
- `DirectoryLoader` + `TextLoader` reads every `.txt` file in `data/text_files/`.
- Each loaded file becomes a **LangChain Document** object containing the text (`page_content`) and metadata like filename, page number, etc.

In [5]:
from langchain_community.document_loaders import PyPDFDirectoryLoader, DirectoryLoader, TextLoader

# Load all PDFs from a folder
pdf_loader = PyPDFDirectoryLoader(PDF_DIR)
pdf_docs = pdf_loader.load()

# Load all .txt files from a folder
txt_loader = DirectoryLoader(TEXT_DIR, glob="*.txt", loader_cls=TextLoader)
txt_docs = txt_loader.load()

# Combine them
all_docs = pdf_docs + txt_docs

print(f"📄 Loaded {len(pdf_docs)} PDF page(s) and {len(txt_docs)} text file(s).")
print(f"📚 Total documents: {len(all_docs)}")

C:\Users\Zeeshan\AppData\Local\Temp\ipykernel_12696\2378799858.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFDirectoryLoader, DirectoryLoader, TextLoader
d:\Projects\Python Projects\RAG Applications\rag-data-ingestion-pipeline\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


📄 Loaded 8 PDF page(s) and 2 text file(s).
📚 Total documents: 10


In [6]:
# Inspect the first document
all_docs[0]

Document(metadata={'producer': 'Microsoft® Word 2016', 'creator': 'Microsoft® Word 2016', 'creationdate': '2020-09-30T14:25:11+03:00', 'author': 'Arbaz Dawood Bhikan', 'moddate': '2020-09-30T14:25:11+03:00', 'source': '..\\data\\pdfs\\BENEFICIARY CAPTURE GUIDLINES.pdf', 'total_pages': 4, 'page': 0, 'page_label': '1'}, page_content='Dear Customer, \n \nKindly follow the below guidelines to enable you in the process of capturing the beneficiary \ndetails. \n \n1. You will receive an email to capture beneficiary bank details along with Authority letter and \nRelease & Discharge form as attachments.  \n \n2. Kindly sign the Release & Discharge / MHB Cash Advance form. Once completed, click the \nbelow link to capture beneficiary bank details. \n \n \n3. Please click “Send OTP” to receive verification code to your registered email address and \nupdate the OTP/ File reference / PNR & Last Name and click on validate.')

---
## Step 2 — ✂️ Text Chunking (Breaking Documents into Pieces)

> **Goal:** Split large documents into small, overlapping pieces that fit within the LLM's context window.

**Why do we chunk?**
- LLMs have a **token limit** — you can't pass an entire 100-page PDF at once.
- Smaller chunks make **retrieval more precise** — finding a relevant paragraph is better than finding a relevant book.

**Why overlap?**
- Overlap ensures that a sentence split across two chunks isn't lost. The end of chunk 1 and the start of chunk 2 share 200 characters, preserving context.

```
Document: "Python is great. It is easy to learn. Many developers love it."

Chunk 1: "Python is great. It is easy to learn."
Chunk 2: "It is easy to learn. Many developers love it."
              ↑ overlap ensures continuity ↑
```

In [7]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,       # Each chunk has max 1000 characters
    chunk_overlap=200      # Chunks overlap by 200 characters
)

chunks = text_splitter.split_documents(all_docs)

print(f"✅ Created {len(chunks)} chunks from {len(all_docs)} document(s).")

✅ Created 12 chunks from 10 document(s).


In [8]:
# Inspect a chunk
print(f"Chunk 0 content ({len(chunks[0].page_content)} chars):")
print(chunks[0].page_content)
print("\nMetadata:", chunks[0].metadata)

Chunk 0 content (576 chars):
Dear Customer, 
 
Kindly follow the below guidelines to enable you in the process of capturing the beneficiary 
details. 
 
1. You will receive an email to capture beneficiary bank details along with Authority letter and 
Release & Discharge form as attachments.  
 
2. Kindly sign the Release & Discharge / MHB Cash Advance form. Once completed, click the 
below link to capture beneficiary bank details. 
 
 
3. Please click “Send OTP” to receive verification code to your registered email address and 
update the OTP/ File reference / PNR & Last Name and click on validate.

Metadata: {'producer': 'Microsoft® Word 2016', 'creator': 'Microsoft® Word 2016', 'creationdate': '2020-09-30T14:25:11+03:00', 'author': 'Arbaz Dawood Bhikan', 'moddate': '2020-09-30T14:25:11+03:00', 'source': '..\\data\\pdfs\\BENEFICIARY CAPTURE GUIDLINES.pdf', 'total_pages': 4, 'page': 0, 'page_label': '1'}


---
## Step 3 — 🔢 Embeddings & Vector Store (Making Text Searchable)

> **Goal:** Convert text chunks into numerical vectors and store them in a database for fast similarity search.

**What are Embeddings?**
- An embedding converts text into a **list of numbers** (a vector) that captures its **meaning**.
- Similar texts produce similar vectors.

**What is ChromaDB?**
- It's a **vector database** — instead of searching by keywords, it searches by **meaning**.
- It saves to disk (`data/vector_store/`), so you don't have to re-process documents every time.

In [9]:
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma

# Load a free, local embedding model
print("🧠 Loading embedding model...")
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

# Store chunks as vectors in ChromaDB
print("💾 Storing vectors in ChromaDB...")
vector_store = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    persist_directory=PERSIST_DIRECTORY
)

print(f"✅ Vector store ready with {vector_store._collection.count()} vectors.")

🧠 Loading embedding model...


C:\Users\Zeeshan\AppData\Local\Temp\ipykernel_12696\1636357822.py:6: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 6595.82it/s]


💾 Storing vectors in ChromaDB...
✅ Vector store ready with 48 vectors.


---
## Step 4 — 🔍 Retriever (Finding Relevant Chunks)

> **Goal:** When a user asks a question, find the most relevant chunks from the vector store.

**How does retrieval work?**
1. The user's question is converted into an embedding (same model as above).
2. ChromaDB compares this embedding against all stored chunk embeddings using **cosine similarity**.
3. The top `k` most similar chunks are returned.

> 💡 This is the **"R" in RAG** — Retrieval!

In [10]:
retriever = vector_store.as_retriever(search_kwargs={"k": 3})

# Example: find 3 most relevant chunks for a query
relevant_docs = retriever.invoke("What is Python?")

print(f"🔍 Found {len(relevant_docs)} relevant chunks:\n")
for i, doc in enumerate(relevant_docs):
    print(f"--- Chunk {i+1} ---")
    print(doc.page_content[:200], "...")
    print(f"Source: {doc.metadata}\n")

🔍 Found 3 relevant chunks:

--- Chunk 1 ---
Python is a high-level, interpreted, general-purpose programming language created by Guido van Rossum and first released in 1991. It is known for its simple, readable syntax, making it one of the easi ...
Source: {'source': '..\\data\\text_files\\python_intro.txt'}

--- Chunk 2 ---
Python is a high-level, interpreted, general-purpose programming language created by Guido van Rossum and first released in 1991. It is known for its simple, readable syntax, making it one of the easi ...
Source: {'source': '..\\data\\text_files\\python_intro.txt'}

--- Chunk 3 ---
Python is a high-level, interpreted, general-purpose programming language created by Guido van Rossum and first released in 1991. It is known for its simple, readable syntax, making it one of the easi ...
Source: {'source': '..\\data\\text_files\\python_intro.txt'}



---
## Step 5 — 🤖 LLM Integration (Generating the Answer)

> **Goal:** Send the retrieved context + user question to an LLM and get a grounded answer.

**What happens here?**
1. **Prompt Template** — We tell the LLM: *"Here's some context from the user's documents. Use it to answer their question."*
2. **Groq + Llama 3** — Groq provides blazing-fast inference for open-source models. The free tier is perfect for learning!
3. **RAG Chain** — LangChain connects the retriever and LLM into a single chain: *Question → Retrieve → Generate → Answer*.

> 💡 This is the **"AG" in RAG** — Augmented Generation!

In [17]:
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

# Initialize the LLM (Groq serves Llama 3 with ultra-fast speed)
llm = ChatGroq(model="llama-3.1-8b-instant", temperature=0.2)

# Create the prompt template
system_prompt = (
    "You are an assistant for question-answering tasks. "
    "Use the following pieces of retrieved context to answer "
    "the question. If you don't know the answer, say that you "
    "don't know.\n\n"
    "Context:\n{context}"
)

prompt = ChatPromptTemplate.from_messages([
    ("system", system_prompt),
    ("human", "{input}"),
])

# Helper to format retrieved docs into a single string
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

# Build the RAG chain using LCEL
rag_chain = (
    {"context": retriever | format_docs, "input": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

print("✅ RAG chain is ready!")


✅ RAG chain is ready!


---
## 💬 Ask a Question!

Now let's test the full pipeline by asking a question about our documents.

In [19]:
# Ask a question!
query = "What is Python and why is it popular?"
print(f"💬 Question: '{query}'\n")
answer = rag_chain.invoke(query)
print(f"💡 Answer:\n{answer}")


💬 Question: 'What is Python and why is it popular?'

💡 Answer:
Python is a high-level, interpreted, general-purpose programming language created by Guido van Rossum and first released in 1991. It is known for its simple, readable syntax, making it one of the easiest languages to learn and one of the most widely used in software development.

Python is popular due to several reasons:

1. Easy to read and write
2. Large standard library
3. Huge ecosystem of third-party packages
4. Cross-platform (Windows, Linux, macOS)
5. Excellent community support
6. Suitable for beginners and professionals alike


In [16]:
# Try another question
query2 = "What are the key features of Java?"
print(f"💬 Question: '{query2}'\n")
answer2 = rag_chain.invoke(query2)
print(f"💡 Answer:\n{answer2}")

💬 Question: 'What are the key features of Java?'

💡 Answer:
The key features of Java are:

1. Platform-independent ("Write Once, Run Anywhere")
2. Object-Oriented Programming (OOP)
3. Strongly and statically typed
4. Automatic memory management (Garbage Collection)
5. Robust exception handling
6. Multithreading support
7. Rich standard library


### Try question from PDFs

In [20]:
# Try another question
query2 = "WHY NRIS CHOOSE FCNR?"
print(f"💬 Question: '{query2}'\n")
answer2 = rag_chain.invoke(query2)
print(f"💡 Answer:\n{answer2}")

💬 Question: 'WHY NRIS CHOOSE FCNR?'

💡 Answer:
According to the provided context, Non-Resident Indians (NRIs) choose FCNR (Fixed Deposit held in a permitted foreign currency) because it offers:

1. Guaranteed tax-free returns
2. Paid in foreign currency
3. Both principal and interest are free to move back to them abroad
4. Their money stays in foreign currency, fully shielded from rupee exchange-rate movements

These benefits make FCNR an attractive option for NRIs living abroad.
